# MIMIC Circulatory Failure Prediction (Bootstrap Trajectories)

Bootstrap trajectory representation only. Compares model and feature configurations across single vs multi-biomarker trajectories and summary statistics.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print('✓ Imports successful')

WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


✓ Imports successful


In [2]:
def _pick_id_col(df):
    for col in ['hadm_id', 'stay_id', 'patientid']:
        if col in df.columns:
            return col
    raise ValueError('No ID column found')


def _pick_time_cols(df):
    if 'time_hours' in df.columns and 'time_hour' in df.columns:
        return 'time_hours', 'time_hour'
    if 'time_days' in df.columns and 'time_day' in df.columns:
        return 'time_days', 'time_day'
    raise ValueError('No time columns found')


BASE_DIR = '../../../results/mimic/circulatory_failure'
PRED_PATH = os.path.join(BASE_DIR, 'circulatory_failure_prediction_dataset.csv')

dataset = pd.read_csv(PRED_PATH)
print(f'✓ Loaded prediction dataset: {len(dataset):,} samples')

lactate_ts = pd.read_csv(os.path.join(BASE_DIR, 'lactate_timeseries.csv'))
heartrate_ts = pd.read_csv(os.path.join(BASE_DIR, 'heartrate_timeseries.csv'))
systolic_ts = pd.read_csv(os.path.join(BASE_DIR, 'systolic_timeseries.csv'))

print('✓ Loaded biomarker time series')
print(f"  Lactate:   {len(lactate_ts):,} rows")
print(f"  Heartrate: {len(heartrate_ts):,} rows")
print(f"  Systolic:  {len(systolic_ts):,} rows")

✓ Loaded prediction dataset: 0 samples
✓ Loaded biomarker time series
  Lactate:   148,446 rows
  Heartrate: 5,903,011 rows
  Systolic:  5,818,845 rows


In [4]:
dataset

,hadm_id,subject_id,admittime,dischtime,los_days,gender,age,hospital_expire_flag,discharge_location,time_hour,...,sodium_mean,spo2_mean,sysbp_mean,tempc_mean,wbc_mean,weight_mean,baseline_lactate,baseline_heartrate,baseline_systolic,target_circulatory_failure


In [ ]:
LOOKBACK_HOURS = 12


def _load_bootstrap_trajectories(biomarker, output_name):
    """
    Load pre-computed bootstrap trajectory probabilities.
    Run the SLURM job first: sbatch scripts/slurm/mimic/run_circulatory_failure_bootstrap.slurm
    Then merge: bash scripts/slurm/mimic/merge_circulatory_failure_bootstrap.sh
    """
    output_path = os.path.join(BASE_DIR, output_name)
    if not os.path.exists(output_path):
        raise FileNotFoundError(
            f"Pre-computed bootstrap file not found: {output_path}\n"
            f"Run the SLURM job first:\n"
            f"  sbatch scripts/slurm/mimic/run_circulatory_failure_bootstrap.slurm\n"
            f"Then merge cohorts:\n"
            f"  bash scripts/slurm/mimic/merge_circulatory_failure_bootstrap.sh"
        )
    
    boot_df = pd.read_csv(output_path)
    print(f"  Loaded {biomarker}: {len(boot_df):,} rows")
    
    # MIMIC uses hadm_id
    id_col = 'hadm_id'
    windowing_col = 'time_hour'
    
    # Rename trajectory columns to standard format
    rename_map = {}
    for col in boot_df.columns:
        if col.endswith('_stable') and not col.startswith(biomarker):
            rename_map[col] = f'{biomarker}_stable'
        elif col.endswith('_gradual') and not col.startswith(biomarker):
            rename_map[col] = f'{biomarker}_gradual'
        elif col.endswith('_rapid') and not col.startswith(biomarker):
            rename_map[col] = f'{biomarker}_rapid'
    
    if rename_map:
        boot_df = boot_df.rename(columns=rename_map)
    
    # Ensure expected columns exist
    expected_cols = [f'{biomarker}_stable', f'{biomarker}_gradual', f'{biomarker}_rapid']
    for col in expected_cols:
        if col not in boot_df.columns:
            boot_df[col] = np.nan
    
    return boot_df[[c for c in [id_col, windowing_col] + expected_cols if c in boot_df.columns]]


print('Loading pre-computed bootstrap trajectory probabilities...')
lactate_boot = _load_bootstrap_trajectories('lactate', 'lactate_trajectory_probs_bootstrap.csv')
heartrate_boot = _load_bootstrap_trajectories('heartrate', 'heartrate_trajectory_probs_bootstrap.csv')
systolic_boot = _load_bootstrap_trajectories('systolic', 'systolic_trajectory_probs_bootstrap.csv')

print('✓ Loaded bootstrap trajectory probabilities')

KeyboardInterrupt: 

In [ ]:
id_col = _pick_id_col(dataset)
_, windowing_col = _pick_time_cols(dataset)

dataset = dataset.merge(lactate_boot, on=[id_col, windowing_col], how='left')
dataset = dataset.merge(heartrate_boot, on=[id_col, windowing_col], how='left')
dataset = dataset.merge(systolic_boot, on=[id_col, windowing_col], how='left')

dataset['lactate_worsening'] = dataset['lactate_gradual'] + dataset['lactate_rapid']
dataset['heartrate_worsening'] = dataset['heartrate_gradual'] + dataset['heartrate_rapid']
dataset['systolic_worsening'] = dataset['systolic_gradual'] + dataset['systolic_rapid']
dataset['multi_marker_mean_worsening'] = dataset[['lactate_worsening', 'heartrate_worsening', 'systolic_worsening']].mean(axis=1)

print(f'✓ Merged bootstrap probabilities: {len(dataset):,} samples')

In [ ]:
def biomarker_summary_stats(ts_df, value_col, lookback_hours=12):
    id_col = _pick_id_col(ts_df)
    time_col, windowing_col = _pick_time_cols(ts_df)
    summary_df_list = []
    for group, group_df in ts_df.groupby(id_col):
        group_df = group_df.sort_values(time_col)
        summary_list = []
        for current_time in group_df[windowing_col].unique():
            window_start = current_time - lookback_hours
            window_data = group_df[group_df[windowing_col].between(window_start, current_time, inclusive='both')]
            if len(window_data) > 0:
                value_mean = window_data[value_col].mean()
                value_max = window_data[value_col].max()
                value_min = window_data[value_col].min()
                value_change = window_data[value_col].iloc[-1] - window_data[value_col].iloc[0]
                value_linear_trend = np.polyfit(window_data[time_col], window_data[value_col], 1)[0] if len(window_data) > 1 else 0
                value_std = window_data[value_col].std() if len(window_data) > 1 else 0
            else:
                value_mean = np.nan
                value_max = np.nan
                value_min = np.nan
                value_change = np.nan
                value_linear_trend = np.nan
                value_std = np.nan
            summary_list.append({
                id_col: group,
                windowing_col: current_time,
                f'{value_col}_mean_{lookback_hours}h': value_mean,
                f'{value_col}_max_{lookback_hours}h': value_max,
                f'{value_col}_min_{lookback_hours}h': value_min,
                f'{value_col}_change_{lookback_hours}h': value_change,
                f'{value_col}_trend_{lookback_hours}h': value_linear_trend,
                f'{value_col}_std_{lookback_hours}h': value_std
            })
        summary_df_list.append(pd.DataFrame(summary_list))
    return pd.concat(summary_df_list, ignore_index=True)

print(f'Computing summary statistics (lookback={LOOKBACK_HOURS} hours)...')
lactate_summary = biomarker_summary_stats(lactate_ts, 'lactate', lookback_hours=LOOKBACK_HOURS)
heartrate_summary = biomarker_summary_stats(heartrate_ts, 'heartrate', lookback_hours=LOOKBACK_HOURS)
systolic_summary = biomarker_summary_stats(systolic_ts, 'systolic', lookback_hours=LOOKBACK_HOURS)

dataset = dataset.merge(lactate_summary, on=[id_col, windowing_col], how='left')
dataset = dataset.merge(heartrate_summary, on=[id_col, windowing_col], how='left')
dataset = dataset.merge(systolic_summary, on=[id_col, windowing_col], how='left')

print('✓ Added summary statistics')

In [ ]:
target_col = next((c for c in ['target_circulatory_failure', 'target_circ_failure', 'target_circulatory'] if c in dataset.columns), None)
if target_col is None:
    raise ValueError('No target column found')

if 'gender' in dataset.columns:
    gender_map = {'M': 1, 'F': 0}
    dataset['gender'] = dataset['gender'].map(gender_map)

static_features = [c for c in ['age', 'gender'] if c in dataset.columns]

exclude_keywords = ['stable', 'gradual', 'rapid', 'worsening', 'marker', 'trend', 'change', 'boot']
dynamic_labs = [
    col for col in dataset.columns
    if any(col.startswith(prefix) for prefix in ['min_', 'mean_', 'max_'])
    and not any(keyword in col.lower() for keyword in exclude_keywords)
 ]
dynamic_vitals = [
    col for col in dataset.columns
    if any(x in col for x in ['heart_rate', 'respiratory_rate', 'o2_sat', 'systolic', 'diastolic', 'mean_bp', 'temperature'])
 ]
dynamic_features = list(dict.fromkeys(dynamic_labs + dynamic_vitals))
static_dynamic_cols = static_features + dynamic_features

lactate_traj = ['lactate_stable', 'lactate_gradual', 'lactate_rapid', 'lactate_worsening']
heartrate_traj = ['heartrate_stable', 'heartrate_gradual', 'heartrate_rapid', 'heartrate_worsening']
systolic_traj = ['systolic_stable', 'systolic_gradual', 'systolic_rapid', 'systolic_worsening']
multi_traj = lactate_traj + heartrate_traj + systolic_traj + ['multi_marker_mean_worsening']

lactate_summary = [c for c in dataset.columns if c.startswith('lactate_') and c.endswith('h')]
heartrate_summary = [c for c in dataset.columns if c.startswith('heartrate_') and c.endswith('h')]
systolic_summary = [c for c in dataset.columns if c.startswith('systolic_') and c.endswith('h')]
multi_summary = lactate_summary + heartrate_summary + systolic_summary

feature_sets = {
    'Single Marker Trajectory': lactate_traj,
    'Multi-Marker Trajectories': multi_traj,
    'Single Marker Summary Stats': lactate_summary,
    'Multi-Marker Summary Stats': multi_summary,
    'Static Only': static_features,
    'Static + Single Marker Trajectory': static_features + lactate_traj,
    'Static + Multi-Marker Trajectories': static_features + multi_traj,
    'Static + Single Marker Summary Stats': static_features + lactate_summary,
    'Static + Multi-Marker Summary Stats': static_features + multi_summary,
    'Static + Single Marker Trajectory + Summary': static_features + lactate_traj + lactate_summary,
    'Static + Multi-Marker Trajectories + Summary': static_features + multi_traj + multi_summary,
}

if len(dynamic_features) > 0:
    feature_sets['Static + Dynamic'] = static_dynamic_cols
    feature_sets['Static + Dynamic + Single Marker Trajectory'] = static_dynamic_cols + lactate_traj
    feature_sets['Static + Dynamic + Multi-Marker Trajectories'] = static_dynamic_cols + multi_traj
    feature_sets['Static + Dynamic + Single Marker Summary Stats'] = static_dynamic_cols + lactate_summary
    feature_sets['Static + Dynamic + Multi-Marker Summary Stats'] = static_dynamic_cols + multi_summary
    feature_sets['Static + Dynamic + Single Marker Trajectory + Summary'] = static_dynamic_cols + lactate_traj + lactate_summary
    feature_sets['Static + Dynamic + Multi-Marker Trajectories + Summary'] = static_dynamic_cols + multi_traj + multi_summary

for set_name, cols in list(feature_sets.items()):
    feature_sets[set_name] = [c for c in cols if c in dataset.columns]

print(f'Feature sets prepared: {len(feature_sets)} total')

In [ ]:
dataset_clean = dataset.dropna(subset=[target_col])
y = dataset_clean[target_col]
groups = dataset_clean[id_col]

traj_cols = [c for c in dataset_clean.columns if any(k in c for k in ['_stable', '_gradual', '_rapid', '_worsening'])]
summary_cols = [c for c in dataset_clean.columns if c.endswith('h') and any(prefix in c for prefix in ['lactate_', 'heartrate_', 'systolic_'])]

dataset_clean[traj_cols] = dataset_clean[traj_cols].fillna(0)
dataset_clean[summary_cols] = dataset_clean[summary_cols].fillna(0)

models_to_evaluate = {
    'XGBoost': lambda pos_weight, seed: XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        scale_pos_weight=pos_weight,
        random_state=seed,
        eval_metric='logloss'
    ),
    'Logistic Regression': lambda pos_weight, seed: LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=seed,
        solver='lbfgs'
    ),
    'Random Forest': lambda pos_weight, seed: RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight='balanced',
        random_state=seed,
        n_jobs=4
    ),
    'Gradient Boosting': lambda pos_weight, seed: HistGradientBoostingClassifier(
        max_bins=225,
        max_depth=3,
        learning_rate=0.1,
        class_weight='balanced',
        random_state=seed
    )
}

n_repeats = 5
n_folds = 5
results = {model_name: {} for model_name in models_to_evaluate.keys()}

for model_name, model_fn in models_to_evaluate.items():
    for feature_set_name, feature_cols in feature_sets.items():
        fold_metrics = {'roc_auc': [], 'avg_precision': [], 'y_true': [], 'y_pred': []}
        for repeat in range(n_repeats):
            shuffle_idx = np.random.RandomState(seed=920+repeat).permutation(len(dataset_clean))
            dataset_repeat = dataset_clean.iloc[shuffle_idx].reset_index(drop=True)
            y_repeat = y.iloc[shuffle_idx].reset_index(drop=True)
            groups_repeat = groups.iloc[shuffle_idx].reset_index(drop=True)
            gkf = GroupKFold(n_splits=n_folds)
            for train_idx, test_idx in gkf.split(dataset_repeat, y_repeat, groups_repeat):
                X_train = dataset_repeat.iloc[train_idx][feature_cols]
                X_test = dataset_repeat.iloc[test_idx][feature_cols]
                y_train = y_repeat.iloc[train_idx]
                y_test = y_repeat.iloc[test_idx]
                imputer = SimpleImputer(strategy='median')
                X_train_imputed = imputer.fit_transform(X_train)
                X_test_imputed = imputer.transform(X_test)
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_imputed)
                X_test_scaled = scaler.transform(X_test_imputed)
                scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
                model = model_fn(scale_pos_weight, 920+repeat)
                model.fit(X_train_scaled, y_train)
                y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
                fold_metrics['roc_auc'].append(roc_auc_score(y_test, y_pred_proba))
                fold_metrics['avg_precision'].append(average_precision_score(y_test, y_pred_proba))
                fold_metrics['y_true'].extend(y_test)
                fold_metrics['y_pred'].extend(y_pred_proba)
        results[model_name][feature_set_name] = fold_metrics

print('✓ Finished model comparisons')

In [ ]:
summary_rows = []
for model_name in models_to_evaluate.keys():
    for name, metrics in results[model_name].items():
        summary_rows.append({
            'Model': model_name,
            'Feature Set': name,
            'ROC-AUC': np.mean(metrics['roc_auc']),
            'ROC-AUC std': np.std(metrics['roc_auc']),
            'AUPR': np.mean(metrics['avg_precision']),
            'AUPR std': np.std(metrics['avg_precision'])
        })

summary_df = pd.DataFrame(summary_rows).sort_values(['Model', 'ROC-AUC'], ascending=[True, False])
summary_df.head(20)

In [ ]:
# Visualize top feature sets per model
for model_name in summary_df['Model'].unique():
    model_df = summary_df[summary_df['Model'] == model_name].copy()
    top_df = model_df.sort_values('ROC-AUC', ascending=False).head(8)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_df, x='ROC-AUC', y='Feature Set', color='#4C72B0')
    plt.title(f"Top Feature Sets by AUROC - {model_name}")
    plt.xlim(0.5, 1.0)
    plt.tight_layout()
    plt.show()

# AUROC boxplots across models
plt.figure(figsize=(12, 6))
sns.boxplot(data=summary_df, x='Model', y='ROC-AUC', color='lightblue')
plt.title('AUROC Distribution Across Feature Sets')
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
print("="*100)
print("MODEL COMPARISON SUMMARY")
print("="*100)

for model_name in summary_df['Model'].unique():
    print(f"\n{model_name}:")
    model_df = summary_df[summary_df['Model'] == model_name]
    display(model_df[['Feature Set', 'ROC-AUC', 'ROC-AUC std', 'AUPR', 'AUPR std']].head(10))

# Best overall per model
print("\n" + "="*100)
print("BEST CONFIGURATION PER MODEL")
print("="*100)
for model_name in summary_df['Model'].unique():
    model_df = summary_df[summary_df['Model'] == model_name]
    best = model_df.sort_values('ROC-AUC', ascending=False).iloc[0]
    print(f"{model_name}: {best['Feature Set']} (AUROC={best['ROC-AUC']:.3f}, AUPR={best['AUPR']:.3f})")